# ZhiXia 分阶段测试

**目的**: 逐模块验证重构后的引擎实现，确保与原功能一致。

**测试流程**:
1. 配置加载验证
2. ASR 引擎测试
3. LLM 引擎测试
4. 输出解析器测试
5. RAG retriever 测试
6. TTS 引擎测试
7. Display 接口测试
8. 完整管线端到端测试

In [ ]:
# Cell 1: 配置加载验证

from pathlib import Path
from tests.test_core import test_config

# 加载配置
success, config, duration, error = test_config()

if success:
    print("✅ 配置加载成功")
    print(f"\n项目根目录: {config.project_root}")
    print(f"LLM 模型路径: {config.llm.model_path}")
    print(f"TTS 模型路径: {config.tts.model_path}")
    print(f"ASR 引擎: {config.asr.engine}")
    print(f"RAG 是否启用: {config.rag.enabled}")
    print(f"结构化输出: {config.llm.enable_structured_output}")

    # 验证配置文件存在
    config_file = config.config_dir / "localconfig.json"
    print(f"\n配置文件: {config_file}")
    print(f"文件存在: {config_file.exists()}")
else:
    print(f"❌ 配置加载失败: {error}")

In [ ]:
# Cell 2: ASR 引擎测试

import logging
from pathlib import Path

from zhixia.config.settings import AppSettings
from zhixia.utils.logging import setup_logging
from tests.test_core import test_asr

# 设置日志
setup_logging("INFO")

config = AppSettings.load()

print("测试 FunASR 引擎...")
success, result, duration, error = test_asr(config)
if success:
    print(f"✅ ASR 测试通过: {result} ({duration:.0f}ms)")
else:
    print(f"❌ ASR 测试失败: {error}")

In [ ]:
# Cell 3: LLM 引擎测试

from tests.test_core import test_llm

config = AppSettings.load()

print("测试 RKLLM 引擎...")
success, result, duration, error = test_llm(config)
if success:
    print(f"✅ LLM 测试通过: {result[:50]}... ({duration:.0f}ms)")
else:
    print(f"❌ LLM 测试失败: {error}")
    print("提示: PC上运行需设置 ZHIXIA_ALLOW_FAKE_LLM=1")

In [ ]:
# Cell 4: 输出解析器测试

from tests.test_core import test_output_parser

print("测试输出解析器...")
success, results, duration, error = test_output_parser()

if success:
    for desc, parsed in results:
        print(f"\n测试 {desc}:")
        print(f"  输出: text={parsed.text!r}, emotion={parsed.emotion!r}")
    print(f"\n✅ 全部通过 ({duration:.0f}ms)")
else:
    print(f"❌ 测试失败: {error}")

In [ ]:
# Cell 5: RAG retriever 测试

from tests.test_core import test_rag

print("测试 RAG retriever...")
success, result, duration, error = test_rag()

if success:
    print(f"✅ RAG 测试通过: {result} ({duration:.0f}ms)")
else:
    print(f"❌ RAG 测试失败: {error}")

In [ ]:
# Cell 6: TTS 引擎测试

from pathlib import Path
from zhixia.config.settings import AppSettings
from tests.test_core import test_tts

config = AppSettings.load()

print("测试 Piper TTS 引擎...")
success, result, duration, error = test_tts(config)

if success:
    print(f"✅ TTS 测试通过: {result} ({duration:.0f}ms)")
else:
    print(f"❌ TTS 测试失败: {error}")

In [ ]:
# Cell 7: Display 接口测试

from tests.test_core import test_display

print("测试 Display 接口...")
success, result, duration, error = test_display()

if success:
    print(f"✅ Display 测试通过 ({duration:.0f}ms)")
else:
    print(f"❌ Display 测试失败: {error}")

print("\n预留 Display 引擎接口：")
print("  - zhixia.display.lcd_display.LCDDisplay")
print("  - zhixia.display.epaper_display.EPaperDisplay")

In [ ]:
# Cell 8: 完整管线端到端测试

from pathlib import Path
from zhixia.config.settings import AppSettings
from tests.test_core import test_pipeline

config = AppSettings.load()

print("准备完整管线测试...")
print(f"测试音频: {config.asr.input_audio}")
print(f"LLM 模型: {config.llm.model_path}")
print(f"TTS 模型: {config.tts.model_path}")
print()

success, result, duration, error = test_pipeline(config)

if success:
    print(f"\n✅ Pipeline 测试通过 ({duration:.0f}ms)")
else:
    print(f"\n❌ Pipeline 测试失败: {error}")
    print("\n要运行完整测试，请确保：")
    print(f"1. 配置文件中的音频存在: {config.asr.input_audio}")
    print(f"2. LLM 模型文件存在: {config.llm.model_path}")
    print(f"3. TTS 模型文件存在: {config.tts.model_path}")
    print(f"4. librkllmrt.so 可访问（在 rknn_libs/ 目录）")

## 测试说明

每个 cell 可以独立运行，测试对应模块的功能。

**注意事项**：
- 开发机上需要安装依赖才能运行测试
- RK3588 上需要确保模型文件和库文件存在
- 模型加载失败是正常的（文件不存在时）
- 关键是验证接口设计和配置加载正确

**下一步**：
1. 运行这些 cell 验证代码结构
2. 在 RK3588 上运行完整测试
3. 根据测试结果调整实现